# Preprocessing and Representations

Purpose: preprocess RNA/protein modalities and build RNA-only PCA, protein-only PCA, and joint RNA+protein PCA representations.

Inputs: `data/processed/pbmc5k_10x_citeseq_imported.h5ad`.

Outputs: `data/processed/pbmc5k_10x_citeseq_representations.h5ad` and candidate target table.

Matching script: `scripts/preprocess_data.py`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "rarecell").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rarecell.config import DATA_DIR, REPRESENTATION_KEY_MAP, TABLES_DIR
from rarecell.io import get_cell_labels, get_protein_matrix, get_rna_adata, load_citeseq, \
    make_candidate_target_population_table, save_citeseq_object
from rarecell.preprocessing import align_cells_between_modalities, preprocess_protein, preprocess_rna
from rarecell.representations import compute_joint_pca_representation, compute_protein_pca, compute_rna_pca
from rarecell.utils import write_json


In [ ]:
INPUT = DATA_DIR / "processed" / "pbmc5k_10x_citeseq_imported.h5ad"
OUTPUT = DATA_DIR / "processed" / "pbmc5k_10x_citeseq_representations.h5ad"
RNA_PCS = 30
PROTEIN_PCS = 10
N_TOP_GENES = 2000
TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
obj = load_citeseq(INPUT)
labels = get_cell_labels(obj)
rna = get_rna_adata(obj)
protein = get_protein_matrix(obj)
rna, protein = align_cells_between_modalities(rna, protein)
if "counts" not in rna.layers:
    rna.layers["counts"] = rna.X.copy()
raw_protein = protein.copy()


In [ ]:
adata = preprocess_rna(rna, n_top_genes=N_TOP_GENES, n_pcs=RNA_PCS)
protein_processed = preprocess_protein(protein).loc[adata.obs_names]
raw_protein = raw_protein.loc[adata.obs_names]

rna_pca = compute_rna_pca(adata, n_components=RNA_PCS)
protein_pca = compute_protein_pca(protein_processed, n_components=PROTEIN_PCS)
joint_pca = compute_joint_pca_representation(rna_pca, protein_pca, scale_blocks=True)

if labels is not None:
    adata.obs["cell_type_simple"] = labels.reindex(adata.obs_names)
adata.obsm["X_rna_pca"] = rna_pca.reindex(adata.obs_names).to_numpy()
adata.obsm["X_protein_pca"] = protein_pca.reindex(adata.obs_names).to_numpy()
adata.obsm[REPRESENTATION_KEY_MAP["joint_pca"]] = joint_pca.reindex(adata.obs_names).to_numpy()
adata.obsm["protein_counts"] = raw_protein.reindex(adata.obs_names).to_numpy()
adata.uns["protein_names"] = list(raw_protein.columns.astype(str))


In [ ]:
candidate_labels = adata.obs["cell_type_simple"] if "cell_type_simple" in adata.obs else adata.obs.get("leiden")
make_candidate_target_population_table(candidate_labels).to_csv(TABLES_DIR / "candidate_target_populations.csv",
                                                                index=False)
write_json({
    "input": str(INPUT),
    "output": str(OUTPUT),
    "representations": list(REPRESENTATION_KEY_MAP),
    "n_cells": int(adata.n_obs),
    "n_rna_features": int(adata.n_vars),
    "n_protein_features": int(raw_protein.shape[1]),
}, TABLES_DIR / "preprocess_run_parameters.json")
save_citeseq_object(adata, OUTPUT)
OUTPUT
